In [ ]:
from fetch_data import fetch_forecasts
import pandas as pd
from utils import build_demand_hours
import joblib
import subprocess

In [ ]:
wind, demand = fetch_forecasts()
prices = pd.read_csv("data/raw/prices_2026-10-30.csv")
oil = 119.20 # price on 28/10 of brent europe https://www.eia.gov/petroleum/prices/table.php
gas = 8.464 # SAP on 28/10 (p/KWh) https://data.nationalgas.com/reports/gas-day-summary
wind = wind[["Datetime_GMT", "Capacity", "Incentive_forecast"]]
demand = demand[["TARGETDATE", "FORECASTDEMAND", "CP_ST_TIME", "CP_END_TIME"]]

In [ ]:
# getting wind into a suitable format for merging with other datasets

wind["Datetime_GMT"] = pd.to_datetime(wind["Datetime_GMT"])
wind_hours = (
    wind.set_index("Datetime_GMT")
        .resample("1h")["Incentive_forecast"]   
        .mean()
        .reset_index()
)
wind_hours["Hour"] = (
    wind_hours["Datetime_GMT"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("Europe/London")
)
wind_hours = wind_hours.drop(columns = ["Datetime_GMT"])

In [ ]:
wind_hours.head()

In [ ]:
# getting demand into a suitable format for merging with other datasets

demand_hours = build_demand_hours(demand, format="%Y%m%d").iloc[:-1]

In [ ]:
# getting prices into a suitable format for merging with other datasets

prices["Hour"] = (
    pd.to_datetime(prices["date"]) + pd.to_timedelta(prices["hour"], unit="h")
).dt.tz_localize("Europe/London", ambiguous="NaT", nonexistent="NaT")

yest_prices_hours = prices[["Hour", "price"]].rename(columns={"price": "Price (GBP/MWhe)"})
yest_prices_hours["Hour"] = yest_prices_hours["Hour"] + pd.DateOffset(days=1)

In [ ]:
oil_hours = pd.DataFrame({
    "Hour": pd.date_range("2026-10-31", periods=24, freq="h", tz="Europe/London"),
    "DCOILBRENTEU": oil,
})
gas_hours = pd.DataFrame({
    "Hour": pd.date_range("2026-10-31", periods=24, freq="h", tz="Europe/London"),
    "sap_lag2d": gas,
})

In [ ]:
def wide(df, col, prefix):
    d = df.dropna(subset=["Hour"]).copy()
    d["date"] = d["Hour"].dt.date
    d["hod"] = d["Hour"].dt.hour
    w = d.pivot(index="date", columns="hod", values=col)
    w.columns = [f"{prefix}_h{h:02d}" for h in w.columns]
    return w

demand = wide(demand_hours, "FORECASTDEMAND", "demand")
wind = wide(wind_hours, "Incentive_forecast", "wind")
oil = wide(oil_hours, "DCOILBRENTEU", "oil")
gas = wide(gas_hours, "sap_lag2d", "gas")
yest_price = wide(yest_prices_hours, "Price (GBP/MWhe)", "price").add_suffix("_lag1d")

In [ ]:
X = (demand.join(wind, how="inner")
           .join(gas, how="inner")
           .join(oil, how="inner")
           .join(yest_price, how="inner"))

X.index = pd.to_datetime(X.index)
X["day_of_week"] = X.index.dayofweek
X["month"] = X.index.month_name()

In [ ]:
ridge = joblib.load("models/ridge.joblib")
forecast = ridge.predict(X)
print(forecast)

In [ ]:
# pushing forecast to github, so that prediction can be made after 9am submission time

forecast_df = pd.DataFrame({
    "hour": range(24),
    "price": ridge.predict(X).flatten()
})

forecast_df.insert(0, "date", X.index[0].strftime("%Y-%m-%d"))
forecast_df.to_csv("forecast.csv", index=False)

subprocess.run(["git", "add", "forecast.csv"], check=True)
subprocess.run(["git", "commit", "-m", "Add price forecast"], check=True)
subprocess.run(["git", "push"], check=True)